# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Physics-Informed PINNs, Deep Ritz Variational Method, Hard Boundary Conditions, and NTK Balancing
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Lagaris, I. E., Likas, A., & Fotiadis, D. I. (1998). *Artificial Neural Networks for Solving Ordinary and Partial Differential Equations*. IEEE TNN.
- E, W., & Yu, B. (2018). *The Deep Ritz Method: A Deep Learning-Based Numerical Algorithm for Solving Variational Problems*. Communications in Mathematics and Statistics.
- Raissi, M., Perdikaris, P., & Karniadakis, G. E. (2019). *Physics-Informed Neural Networks: A Deep Learning Framework for Solving Forward and Inverse Problems*. Journal of Computational Physics.
- Wang, S., Teng, Y., & Perdikaris, P. (2021). *Understanding and Mitigating Gradient Pathologies in Physics-Informed Neural Networks*. SIAM Journal on Scientific Computing.

## 2. Mathematical Foundations & Governing Equations

### Strong-Form PINN Residual (1D Poisson)
$$-\frac{d^2 u}{dx^2} = f(x) \implies r(x) = -\frac{d^2 u}{dx^2} - \pi^2 \sin(\pi x), \quad \mathcal{L}_{\text{PDE}} = \frac{1}{N} \sum_{i=1}^N |r(x_i)|^2$$

### Lagaris Hard Boundary Condition Ansatz ($u(0)=0, u(1)=0$)
$$u_{\text{hard}}(x) = x(1 - x) \cdot u(x), \quad \frac{du_{\text{hard}}}{dx} = (1 - 2x)u(x) + x(1 - x)\frac{du}{dx}$$

### Deep Ritz Variational Energy
$$\mathcal{E}(u) = \frac{1}{N}\sum_{i=1}^N \left( \frac{1}{2} \left|\frac{du_{\text{hard}}}{dx}(x_i)\right|^2 - f(x_i) u_{\text{hard}}(x_i) \right)$$

### NTK Eigenvalue Gradient Balancing
$$\lambda_{\text{NTK}} = \frac{\|\nabla_{w_2} \mathcal{L}_{\text{PDE}}\|_2}{\|\nabla_{w_2} \mathcal{L}_{\text{BC}}\|_2 + \epsilon}, \quad \nabla_{\text{balanced}} = g_{\text{pde}} + \lambda_{\text{NTK}} g_{\text{bc}}$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
from safetensors.torch import load_file

torch.manual_seed(42)

# Collocation coordinates in (0, 1)
x_colloc = torch.linspace(0.1, 0.9, 8).unsqueeze(1).requires_grad_(True)
w1 = torch.randn(1, 16) * 0.5
b1 = torch.randn(16) * 0.1
w2 = (torch.randn(16, 1) * 0.5).requires_grad_(True)
b2 = torch.randn(1) * 0.1

print(f"Collocation points: {x_colloc.shape}, Weights: w1 {w1.shape}, w2 {w2.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
# Forward pass
z = x_colloc @ w1 + b1
a = torch.tanh(z)
u = a @ w2 + b2

# Derivatives via autograd
grad_ones = torch.ones_like(u)
u_x_auto = torch.autograd.grad(u, x_colloc, grad_outputs=grad_ones, create_graph=True)[0]
u_xx_auto = torch.autograd.grad(u_x_auto, x_colloc, grad_outputs=grad_ones, create_graph=True)[0]

# Analytical derivatives
psi = (1.0 - a**2) * w1
u_x_ana = psi @ w2
phi = (-2.0 * a * (1.0 - a**2)) * (w1**2)
u_xx_ana = phi @ w2

# PDE residual
f_source = (math.pi**2) * torch.sin(math.pi * x_colloc)
pde_residual = -u_xx_ana - f_source
loss_pde = torch.mean(pde_residual**2)

# Hard boundary condition
d = x_colloc * (1.0 - x_colloc)
d_x = 1.0 - 2.0 * x_colloc
u_hard = d * u
u_hard_x = d_x * u + d * u_x_ana

# Deep Ritz Energy
ritz_energy = torch.mean(0.5 * (u_hard_x**2) - f_source * u_hard)

# Soft BC & NTK gradient balancing
x_bc = torch.tensor([[0.0], [1.0]])
z_bc = x_bc @ w1 + b1
a_bc = torch.tanh(z_bc)
u_bc = a_bc @ w2 + b2
loss_bc = torch.mean(u_bc**2)

g_pde = (-2.0 / x_colloc.shape[0]) * (phi.t() @ pde_residual)
g_bc = (2.0 / x_bc.shape[0]) * (a_bc.t() @ u_bc)
norm_pde = torch.norm(g_pde)
norm_bc = torch.norm(g_bc)
lambda_ntk = norm_pde / (norm_bc + 1e-8)
g_balanced = g_pde + lambda_ntk * g_bc

print(f"PDE Loss: {loss_pde.item():.4f}, Ritz Energy: {ritz_energy.item():.4f}, Lambda NTK: {lambda_ntk.item():.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Autograd vs Closed-form 1st and 2nd derivatives
diff_x = torch.max(torch.abs(u_x_auto - u_x_ana)).item()
diff_xx = torch.max(torch.abs(u_xx_auto - u_xx_ana)).item()
assert diff_x < 1e-6, f"1st derivative discrepancy: {diff_x:.2e}"
assert diff_xx < 1e-6, f"2nd derivative discrepancy: {diff_xx:.2e}"

# Invariant 2: Hard boundary condition exact vanishing at boundaries
x_zero = torch.tensor([[0.0]])
x_one = torch.tensor([[1.0]])
u_h_zero = (x_zero * (1.0 - x_zero)) * (torch.tanh(x_zero @ w1 + b1) @ w2 + b2)
u_h_one = (x_one * (1.0 - x_one)) * (torch.tanh(x_one @ w1 + b1) @ w2 + b2)
assert torch.allclose(u_h_zero, torch.zeros_like(u_h_zero)), "Hard BC failed at x=0"
assert torch.allclose(u_h_one, torch.zeros_like(u_h_one)), "Hard BC failed at x=1"

# Invariant 3: NTK gradient balancing equalizes gradient contributions
scaled_bc_norm = torch.norm(lambda_ntk * g_bc).item()
assert abs(norm_pde.item() - scaled_bc_norm) < 1e-4, "NTK balancing did not match gradient scales"
print("All mathematical invariants (autograd parity, hard BC vanishing, NTK scale matching) verified.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = ((1.0 - torch.tanh(x_colloc @ w1 + b1)**2) * w1) @ w2
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"Analytical PINN derivative step latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: Using ReLU activation in strong-form 2nd-order PINN
u_relu = torch.relu(x_colloc @ w1 + b1) @ w2 + b2
u_x_relu = torch.autograd.grad(u_relu, x_colloc, grad_outputs=torch.ones_like(u_relu), create_graph=True)[0]
u_xx_relu = torch.autograd.grad(u_x_relu, x_colloc, grad_outputs=torch.ones_like(u_x_relu))[0]
# 2nd derivative of ReLU is identically zero almost everywhere
laplacian_vanished = torch.all(u_xx_relu == 0.0).item()
print(f"Pathological: ReLU 2nd derivative vanished identically to zero: {laplacian_vanished}")

# Corrected: Smooth C-infinity activation (tanh) yields well-conditioned non-zero Laplacian
print(f"Corrected: tanh 2nd derivative non-zero norm: {torch.norm(u_xx_ana).item():.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Order Halving via Deep Ritz**: Formulating PDEs through their Dirichlet variational energy cuts autograd graph depth in half by avoiding second-order derivatives.
2. **Zero-Violation Boundary Guarantee**: Lagaris hard boundary ansätze eliminate boundary penalty hyperparameter tuning and guarantee exact physics adherence by construction.
3. **Mitigating Spectral Stiffness**: Dynamic NTK eigenvalue balancing prevents stiff PDE residuals from overwhelming soft boundary penalties during gradient descent.